# From SFE downfolded self-energy in tbtrans 
# to TBTGF file

### The idea:
1) First do tbtrans on the big system and save a self-energy (*SE.nc* file )in the scattering region
2) Then read this and make a smaller system including just a fraction of the left/right couplings
3) Write the *TBTGF*-files to be included in a new smaller tbtrans calculation: This could then be reused in tbtrans calculations on smaller systems where we avoid using large electrode parts

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
#import matplotlib.tri as tri
import sisl as si
import os
from tqdm.auto import tqdm
import time
from scipy.spatial import cKDTree
from scipy.linalg import block_diag
from pathlib import Path

In [ ]:
from ase.visualize import view
from ase.visualize import plot

In [ ]:
def xzplot(geom):
    plot.plot_atoms(geom.to.ase(),rotation='90y')

In [ ]:
lbl = "DLys"

In [ ]:
# Original large Hamiltonian and geometry
H0 = si.get_sile(lbl+".TSHS").read_hamiltonian()
g = si.get_sile(lbl+".XV").read_geometry()

In [ ]:
view(g.to.ase())

## Smaller structure: Set length of parts to include in left and right electrodes

In [ ]:
LLeft = 4.   # include this part of left lead
LRight = 4. 

In [ ]:
z = g.xyz.T[2] ## assume the structure is along z
mol_idx = np.nonzero(g.atoms.Z!=79)[0]
zmin_mol = min(z[mol_idx])
zmax_mol = max(z[mol_idx])

lead_idx  = np.nonzero(g.atoms.Z==79)[0]
left_idx  = lead_idx[z[lead_idx] < zmin_mol]
right_idx = lead_idx[z[lead_idx] > zmax_mol]

In [ ]:
atoms_device_idx = np.nonzero((z > max(z[left_idx]) - LLeft)  &  (z < min(z[right_idx]) + LRight) )[0]

In [ ]:
print("Orig. na = ",g.na, ", Downfolded na=",len(atoms_device_idx))

In [ ]:
xzplot(g.sub(atoms_device_idx))

In [ ]:
view(g.to.ase())

### Write file to include in tbtrans to get SFE file for the small region

In [ ]:
def write_fdf_device_indices_grouped(indices):
    # Group consecutive indices into sequences
    sequences = []
    current_sequence = [indices[0]]

    for i in range(1, len(indices)):
        if indices[i] == indices[i - 1] + 1:  # Check if the current index is consecutive
            current_sequence.append(indices[i])
        else:
            sequences.append(current_sequence)
            current_sequence = [indices[i]]

    # Append the last sequence
    sequences.append(current_sequence)
    # Convert sequences into FDF-compatible ranges
    fdf_lines = []
    for seq in sequences:
        if len(seq) > 1:
            fdf_lines.append(f"  atom [{seq[0]+1} -- {seq[-1]+1}]")
        else:
            fdf_lines.append(f"  atom {seq[0]+1}")

    # Combine into the FDF block
    fdf_block = "%block TBT.Atoms.Device\n" + "\n".join(fdf_lines) + "\n%endblock"

    # Print the FDF block
    print(fdf_block)
    # Specify the output file name
    output_file = "TBT_Atoms_Device.fdf"

    # Write the FDF block to the file
    with open(output_file, "w") as f:
        f.write(fdf_block)
    print(f"FDF block written to {output_file}")

In [ ]:
with open("TBT_SFE.fdf", "w") as f:
    f.write(f"""
%include TBT_Atoms_Device.fdf
TBT.CDF.Compress 9  ### maybe higher.. 9?
TBT.CDF.SelfEnergy.Save
TBT.CDF.SelfEnergy.Precision single
TBT.CDF.SelfEnergy.Save.Mean  False
TBT.SelfEnergy.Save.Mean False
TBT.SelfEnergy.Save True\n""")
write_fdf_device_indices_grouped(atoms_device_idx)


# >> Now run tbtrans
### Now after running tbtrans read the SFE file and transform to TBTGF file for the left/right GF's for the new smaller structure 

In [ ]:
tbt_dir = "."
SEfile = tbt_dir +"/siesta.TBT.SE.nc"
tbtse = si.get_sile(SEfile)
En = tbtse.E
kpts = tbtse.k
wk = tbtse.wk
eta = tbtse.eta()
a_dev = tbtse.a_dev
elecs = tbtse.elecs
H_dev = H0.sub(a_dev)
print("Device atoms: ",a_dev,": ",H_dev.na)
for elec in elecs:
    print("Electrode:",elec)
    pvt = tbtse.pivot(elec=elec, in_device=True, sort=True).reshape(-1, 1) 
    elec_atoms = np.unique(H_dev.geometry.o2a(pvt).flatten())
    print("atoms: ",elec_atoms,": ",len(elec_atoms))

In [ ]:
view(g.sub(a_dev).to.ase())
view(g.to.ase())

## Now we write new TBTGF files for the small calculation

In [ ]:
si.io.table.tableSile(dir+'/contour.IN', 'w').write_data(En, np.zeros(En.size) + eta)

New smaller H

In [ ]:
H_dev.write("H_dev.TSHS")

for elec in tbtse.elecs:
    pvt = tbtse.pivot(elec=elec, in_device=True, sort=True).reshape(-1, 1) 
    elec_atoms = np.unique(H_dev.geometry.o2a(pvt).flatten())
    H_elec = H_dev.sub(elec_atoms)
    bz = si.BrillouinZone(H_elec)
    #print(H_elec.no)
    H_elec.write("H_"+elec+".TSHS")
    with si.io.tbtgfSileTBtrans(dir+"/"+elec+".TBTGF") as f:
        f.write_header(bz, En + eta*1j, mu=0.0)
        for ispin1, new_k, k1, E1 in f:
            tmp = tbtse.self_energy(elec=elec,E=E1,k=k1,sort=True).data
            #print(tmp.shape)
            if new_k:
                Sk = (H_elec.Sk()).todense()
                Hk = (H_elec.Hk()).todense()
                f.write_hamiltonian(Hk,Sk)
            Sigma = np.zeros_like(Sk, dtype=np.complex128)
            Sigma += tmp
            SeHSE = Sk*(E1+eta*1j)-Hk-Sigma
            f.write_self_energy(SeHSE)

### Write new smaller device

In [ ]:
with open(dir+"/RUNTBT-smaller.fdf", "w") as f:
    f.write(f"""
SystemName siesta
SystemLabel stbt
TBT.k [1 1 1]
TBT.Contours.Eta  {eta} eV
TBT.Elecs.Eta   {eta} eV

%block TBT.contour.line
 from -2000. eV to 2000. eV  # write whatever
 file contour.IN
%endblock

TBT.DOS.A.All True
#TBT.CDF.SelfEnergy.Save

TBT.HS ./H_dev.TSHS
%include ELEC-smaller.fdf
#%include TBT_Atoms_Device.fdf\n
""")

with open(dir+"/ELEC-smaller.fdf", "w") as f:
    # Write the list of electrodes
    f.write("%block TS.Elecs\n")
    for elec in elecs:
        f.write(f"  {elec}\n")
    f.write("%endblock TS.Elecs\n\n")
    
    elec = 'Left'
    f.write(f"""%block TS.Elec.{elec}
HS ./H_{elec}.TSHS
Bulk False 
semi-inf-direction -c
electrode-position {1}
Out-of-core true
tbt.Gf {elec}.TBTGF
%endblock TS.Elec.{elec}\n
    """)
    elec = 'Right'
    f.write(f"""
%block TS.Elec.{elec}
HS ./H_{elec}.TSHS
Bulk False
semi-inf-direction +c
electrode-position end {-1}
Out-of-core true
tbt.Gf {elec}.TBTGF
%endblock TS.Elec.{elec}\n
    """)

## Check that the transmissions are the same:

In [ ]:
dir = "."
tbtout = si.io.tbtrans.tbtncSileTBtrans(dir+"/siesta.TBT.nc")
stbtout = si.io.tbtrans.tbtncSileTBtrans(dir+"/stbt.TBT.nc")

elecs = tbtout.elecs
En1 = tbtout.E
for i in range(len(elecs)):
    for j in range(i):
        plt.plot(En1,tbtout.transmission(elec_from = elecs[j], elec_to = elecs[i]),label=elecs[i]+"-to-"+elecs[j]);
        plt.plot(En1,stbtout.transmission(elec_from = elecs[j], elec_to = elecs[i]),'o',label=elecs[i]+"-to-"+elecs[j]);
        #plt.plot(En,tdat,'X')
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5));
plt.tight_layout()